In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

loadsteps = true
savesteps = !loadsteps
loadsteps, savesteps

In [ ]:
# Varying proportionality constant.
ρlist = Vector{defFloat}( 10.0.^(-2:0.25:1) )
K = length( ρlist )

# System size and environment density.
μ = 1.0
N = 100
L = √(N/μ)

# Activity transition variables.
δt = Δt
η = 1/500;  β = 1.0;  γ = 1/10;  τ = 75.0

# Movement variables.
ξ = 0.1;  λ = 0.50;  ϕ = 0.25
s = 10.0/ξ^(ϕ + 1)

# Generate parameter variable.
dparams = Params(; ρ=0.1, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=s )
dscale = Scale( 1.0, ξ )

# Generate parameter lists.
paramslist = [adjparams( dparams; ρ=ρ ) for ρ ∈ ρlist]
nondimlist = [Nondim( params; scale=dscale ) for params ∈ paramslist];

In [ ]:
case = "../data/results/case-1_crit-rho/"
if true
    if !isdir( case )
        mkpath( case )
    end
    saveparams( case*"default-params.json", dparams )
    savescale( case*"default-scale.json", dscale )
end

# Data folder name.
folderlist = [
    case*"N-$(N)/spd-$(round( s, digits=6 ) )/rho-$(round( ρ, digits=6 ))/"
    for ρ ∈ ρlist]

In [ ]:
# Run simulation under each environment parameter.
T = round( defInt, 500/dscale.T );  M = 50
Tload = T  # If applicable.
Nt = round( defInt, T/δt );  tlist = 1:Nt
nt = round( defInt, 1/(2*δt*dscale.T) );  tsave = Set( 1:nt:Nt );

# Frequency of adjacency calculation.
δt̂ = round( defInt, 0.1/δt );

In [ ]:
# Initialize list and run optimization.
xdatalist = [[Matrix{defFloat}( undef, length( tsave ) + 1, 3 ) for _ ∈ 1:M] for _ ∈ 1:K]
zdata = Matrix{State}( undef, K, M )
@threads for k ∈ 1:K
    nondim = nondimlist[k]
    for m ∈ 1:M
        # If steps are already saved, use as initial state.
        file = loadsteps ? folderlist[k]*"steps/state_T-$(Tload)_m-$(m).txt" : nothing

        # Initialize agent states.
        z = initialstate( N, L; A=1, file=file )
        ẑ = copystate( z )

        # Initialize adjacency and saved state.
        A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ )
        xdatalist[k][m][1,:] = statecomposition( N, ẑ )

        # Run simulation.
        t̂ = 2
        for t ∈ tlist
            # Update the adjacency matrix.
            (t % δt̂) == 0 && (A = proximity( N, L, nondim.r, nondim.α, z.x, z.y, z.θ ))

            # Step simulation.
            step!( N, L, nondim, z, ẑ; A=A, δt=δt )

            # Save state if in appropriate subset.
            t ∈ tsave && (xdatalist[k][m][t̂,:] = statecomposition( N, ẑ ); t̂ += 1)

            # Swap contents.
            tmp = z;  z = ẑ;  ẑ = tmp
        end

        # Save last simulation state.
        zdata[k,m] = z
    end
end

In [ ]:
# Compute determinism metric and related statistics.
Rdata = Matrix{RecurrenceMap}( undef, K, M )
ςdata = Matrix{defFloat}( undef, K, M )
for k ∈ 1:K
    @threads for m ∈ 1:M
        Rdata[k,m] = recurrence( xdatalist[k][m]; δx=1/100 )
        ςdata[k,m] = determinism( Rdata[k,m]; ℓ0=15 )
    end
end

# Determinism statistics.
ς̄list = vcat( mean( ςdata, dims=2 )... )
ς̄stnd = vcat( std(  ςdata, dims=2 )... );

In [ ]:

# Plot the determinism as a function of the system size.
plt = plot( size=(300,200), xformatter=:plain, dpi=600 )

ρc = criticalρ( nondimlist[1], N, μ )
plot!( plt, [ρc, ρc], [0, 1]; color=:gray, lw=2 )
plot!( plt, ξ*ρlist, ς̄list; ribbon=ς̄stnd, color=:black, fillalpha=1/4, lw=2, marker=:circ )

plot!( plt; xlims=(ξ*ρlist[1],ξ*ρlist[end]), xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="coefficient of speed, "*L"s_0", ylabel="determinism", legend=false )

# saveplot( plt, figurefolder*"" )

In [ ]:
k = 1

# Plot the mean activity deries for each duration value.
plt = plot( size=(400,200), xformatter=:plain, margin=10pt, dpi=600 )

for m ∈ 1:M
    alist = xdatalist[k][m][:,1]
    if m == M
        plot!( plt, δt*(0:nt:Nt), alist; color=:cornflowerblue, alpha=1, lw=2, label="" )
    else
        plot!( plt, δt*(0:nt:Nt), alist; color=:black, alpha=1/6, label="" )
    end
end

plot!( plt; xlims=(0,T), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1), ylabel="proportion of\nants active, "*L"a" )

In [ ]:
for (k, folder) ∈ enumerate( folderlist )
    if !isdir( folder*"steps/" )
        mkpath( folder*"steps/" )
    end

    if true
        # Unpack save parameters.
        xdata = xdatalist[k]
        params = paramslist[k]

        # Save macroscopic measurements.
        writedlm( folder*"activity_T-$(round( defInt, T )).txt", [xlist[:,1] for xlist ∈ xdata] )
        writedlm( folder*"inactivity_T-$(round( defInt, T )).txt", [xlist[:,2] for xlist ∈ xdata] )
        writedlm( folder*"refractory_T-$(round( defInt, T )).txt", [xlist[:,3] for xlist ∈ xdata] )
        writedlm( folder*"determinism_T-$(round( defInt, T )).txt", ςdata[k,:] )

        # Save dimensional parameters and scale.
        saveparams( folder*"params.json", params )
        savescale( folder*"scale.json", dscale )
    end

    if savesteps
        for m ∈ 1:M
            savestate( folder*"steps/state_T-$(T)_m-$(m).txt", zdata[k,m] )
        end
    end
end